In [0]:
# ===================================================
# BLOCK 1 — IMPORTS AND CONFIGURATION
# ===================================================  

from pyspark.sql import functions as F


"""
Validate persisted Silver reference models, hierarchy relationships,
analytical data types, production-lot reference integrity, and repeatable
pipeline behavior.
"""

SILVER_TABLES = {
    "product_groups": "semiconplus_portfolio.silver.product_groups",
    "devices": "semiconplus_portfolio.silver.devices",
    "sites": "semiconplus_portfolio.silver.sites",
    "equipment": "semiconplus_portfolio.silver.equipment",
}

EXPECTED_COUNTS = {
    "product_groups": 6,
    "devices": 30,
    "sites": 3,
    "equipment": 24,
}

QUARANTINE_TABLE = (
    "semiconplus_portfolio.quarantine.reference_records"
)
PRODUCTION_LOTS_TABLE = (
    "semiconplus_portfolio.silver.production_lots"
)
QUALITY_TABLE = (
    "semiconplus_portfolio.monitoring.data_quality_results" 
)

In [0]:
# ===================================================
# BLOCK 2 — TABLE AND COUNT VALIDATION
# ===================================================  

"""
Confirm that every reference output exists and retains its approved
initial-load population.
"""

for entity_name, table_name in SILVER_TABLES.items():
    assert spark.catalog.tableExists(table_name)

    row_count = spark.table(table_name).count()
    expected_count = EXPECTED_COUNTS[entity_name]

    print(f"{entity_name}: {row_count}")
    assert row_count == expected_count

assert spark.catalog.tableExists(QUARANTINE_TABLE)
assert spark.table(QUARANTINE_TABLE).count() == 0

print("Silver reference table-count validation passed.")

In [0]:
# ===================================================
# BLOCK 3 — BUSINESS-KEY VALIDATION
# ===================================================  

"""
Confirm that each Silver reference contains complete and unique business
keys.
"""

reference_keys = {
    "product_groups": "product_group_id",
    "devices": "device_id",
    "sites": "site_id",
    "equipment": "equipment_id",
}

for entity_name, key_column in reference_keys.items():
    table_df = spark.table(SILVER_TABLES[entity_name])

    invalid_key_count = table_df.filter(
        F.col(key_column).isNull()
        | (F.trim(F.col(key_column)) == "")
    ).count()

    duplicate_key_count = (
        table_df
        .groupBy(key_column)
        .count()
        .filter(F.col("count") > 1)
        .count()
    )

    print(
        f"{entity_name}: invalid={invalid_key_count}, "
        f"duplicates={duplicate_key_count}"
    )

    assert invalid_key_count == 0
    assert duplicate_key_count == 0

print("Silver reference business-key validation passed.")

In [0]:
# ===================================================
# BLOCK 4 — HIERARCHY VALIDATION
# ===================================================  
"""
Confirm that every device resolves to one product group and every piece
of equipment resolves to one manufacturing site.
"""

devices_df = spark.table(SILVER_TABLES["devices"])
product_groups_df = spark.table(SILVER_TABLES["product_groups"])
equipment_df = spark.table(SILVER_TABLES["equipment"])
sites_df = spark.table(SILVER_TABLES["sites"])

unknown_device_product_groups = (
    devices_df
    .join(
        F.broadcast(product_groups_df.select("product_group_id")),
        "product_group_id",
        "left_anti",
    )
    .count()
)

unknown_equipment_sites = (
    equipment_df
    .join(
        F.broadcast(sites_df.select("site_id")),
        "site_id",
        "left_anti",
    )
    .count()
)

print(f"Unknown device product groups: {unknown_device_product_groups}")
print(f"Unknown equipment sites: {unknown_equipment_sites}")

assert unknown_device_product_groups == 0
assert unknown_equipment_sites == 0

print("Silver reference hierarchy validation passed.")

In [0]:
# ===================================================
# BLOCK 5 — ATTRIBUTE AND DATA-TYPE VALIDATION
# ===================================================  

"""
Confirm that security, yield-target, and equipment-capacity attributes
use the approved analytical data types and value ranges.
"""

assert dict(product_groups_df.dtypes)["restricted"] == "boolean"
assert dict(devices_df.dtypes)["target_yield"] == "decimal(9,6)"
assert dict(equipment_df.dtypes)["rated_units_per_hour"] == "int"

assert devices_df.filter(
    (F.col("target_yield") < 0)
    | (F.col("target_yield") > 1)
).count() == 0

assert equipment_df.filter(
    F.col("rated_units_per_hour") <= 0
).count() == 0

assert sites_df.filter(
    F.col("timezone").isNull()
    | (F.trim(F.col("timezone")) == "")
).count() == 0

print("Silver reference attribute validation passed.")

In [0]:
# ===================================================
# BLOCK 6 — PRODUCTION-LOT REFERENCE VALIDATION
# ===================================================  

"""
Confirm that all accepted Silver production lots resolve to consistent
device, product-group, site, and equipment references.
"""

lots_df = spark.table(PRODUCTION_LOTS_TABLE).alias("lots")

reference_failures_df = (
    lots_df
    .join(
        F.broadcast(
            devices_df.select("device_id", "product_group_id").alias("d")
        ),
        F.col("lots.device_id") == F.col("d.device_id"),
        "left",
    )
    .join(
        F.broadcast(sites_df.select("site_id").alias("s")),
        F.col("lots.site_id") == F.col("s.site_id"),
        "left",
    )
    .join(
        F.broadcast(
            equipment_df.select("equipment_id", "site_id").alias("e")
        ),
        (F.col("lots.equipment_id") == F.col("e.equipment_id"))
        & (F.col("lots.site_id") == F.col("e.site_id")),
        "left",
    )
    .filter(
        F.col("d.device_id").isNull()
        | F.col("s.site_id").isNull()
        | F.col("e.equipment_id").isNull()
        | (F.col("lots.product_group_id") != F.col("d.product_group_id"))
    )
    .select("lots.*")
)

reference_failure_count = reference_failures_df.count()

display(reference_failures_df.limit(20))

assert reference_failure_count == 0

print("Silver production-lot reference validation passed.")

In [0]:
# ===================================================
# BLOCK 7 — QUALITY-RESULT VALIDATION
# =================================================== 

"""
Confirm that monitoring contains successful results for every reference
dataset and the production-lot relationship check.
"""

assert spark.catalog.tableExists(QUALITY_TABLE)

quality_df = spark.table(QUALITY_TABLE)

expected_datasets = [
    "product_groups",
    "devices",
    "sites",
    "equipment",
    "production_lot_reference_integrity",
]

for dataset_name in expected_datasets:
    successful_count = quality_df.filter(
        (F.col("dataset_name") == dataset_name)
        & (F.col("validation_status") == "PASSED")
    ).count()

    assert successful_count >= 1

display(
    quality_df
    .filter(F.col("dataset_name").isin(expected_datasets))
    .orderBy(F.col("validated_at_utc").desc())
    .limit(20)
)

print("Silver quality-result validation passed.")

In [0]:
# ===================================================
# BLOCK 8 — CAPTURE RERUN BASELINE
# =================================================== 

"""
Capture the current Silver reference counts before repeating the
deterministic snapshot transformation.
"""

counts_before_rerun = {
    entity_name: spark.table(table_name).count()
    for entity_name, table_name in SILVER_TABLES.items()
}

quarantine_before_rerun = spark.table(QUARANTINE_TABLE).count()

print(counts_before_rerun)
print(f"Reference quarantine: {quarantine_before_rerun}")

In [0]:

# ---------------
# RERUN PROCEDURE
# ---------------

# 1. Run Block 8.
# 2. Rerun all blocks in 07_silver_reference_models.
# 3. Return here without clearing the Python session.
# 4. Run Block 9.

# ===================================================
# BLOCK 9 — IDEMPOTENCY VALIDATION
# =================================================== 


"""
Confirm that repeating the reference transformation replaces existing
snapshots without changing record counts or creating duplicate keys.
"""

counts_after_rerun = {
    entity_name: spark.table(table_name).count()
    for entity_name, table_name in SILVER_TABLES.items()
}

quarantine_after_rerun = spark.table(QUARANTINE_TABLE).count()

assert counts_after_rerun == counts_before_rerun
assert quarantine_after_rerun == quarantine_before_rerun

for entity_name, key_column in reference_keys.items():
    duplicate_count = (
        spark.table(SILVER_TABLES[entity_name])
        .groupBy(key_column)
        .count()
        .filter(F.col("count") > 1)
        .count()
    )
    assert duplicate_count == 0

print("SILVER REFERENCE IDEMPOTENCY TEST PASSED")

In [0]:

# ===================================================
# BLOCK 10 — FINAL VALIDATION RESULT
# =================================================== 

"""
Publish the final persisted-reference validation result for execution
evidence and project documentation.
"""

print("SILVER REFERENCE VALIDATION PASSED")
print("Product groups: 6")
print("Devices: 30")
print("Sites: 3")
print("Equipment: 24")
print("Hierarchy failures: 0")
print("Production-lot reference failures: 0")
print("Reference quarantine records: 0")